# Test it locally first

In [1]:
# Check if paddle is correctly setup
import paddle
# print(paddle.__version__)
# print(paddle.device.get_device())
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


c:\Users\lhaus\Documents\FH\SDC\Digital-Shelf\.venv\Lib\site-packages\paddle\pir\math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


In [2]:
import cv2 as cv
from pathlib import Path

img_path = Path("../data/sample_images") / "PXL_20251023_095235670.jpg"
img = cv.imread(img_path)

In [10]:
# Initialize PaddleOCR instance
from paddleocr import PaddleOCR
ocr = PaddleOCR(
    use_doc_orientation_classify=True,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

# Run OCR inference on a sample image 
result = ocr.predict(
    input=img)

# Visualize the results and save the JSON results
for res in result:
    res.print()
    res.save_to_img("output")
    res.save_to_json("output")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lhaus\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lhaus\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lhaus\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lhaus\.paddlex\official_models\PP-OCRv5_server_rec`.
Resized image size (8160x6144) exceeds max_side_limit of 4000. Resizing to fit within limit.
{'res': {'input_path': None, 'page_index': None, 'mod

# Testing the local OCR server

In [98]:
import base64
import json
from pathlib import Path
import requests
img_path = Path("../data/sample_images") / "PXL_20251023_081854125.jpg"

with open(file=img_path, mode="rb") as f:
    base64_img = base64.b64encode(f.read()).decode("utf-8")
request_dict = {
  "file": base64_img,
  "fileType": 1,
  "useDocOrientationClassify": True,
  "useDocUnwarping": True,
  "useTextlineOrientation": True,
  "textDetLimitSideLen": 960,
  "textDetLimitType": "max",
  "textDetThresh": 0.3,
  "textDetBoxThresh": 0,
  "textDetUnclipRatio": 1.6,
  "textRecScoreThresh": 0,
  "visualize": True
}
headers = {"Content-Type": "application/json"}

response = requests.post(
    url="http://192.168.178.52:8080/ocr",
    json=request_dict,
    headers=headers
)

response.raise_for_status()

In [99]:
res = response.json()
print(str(res)[:2000])

{'logId': '273d1ee3-b8b6-4e00-ae37-0aa9a49d5b24', 'result': {'ocrResults': [{'prunedResult': {'model_settings': {'use_doc_preprocessor': True, 'use_textline_orientation': True}, 'doc_preprocessor_res': {'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 180}, 'dt_polys': [[[1028, 2519], [3724, 2327], [3759, 2837], [1063, 3029]], [[1121, 3163], [3818, 2952], [3838, 3216], [1141, 3427]], [[1142, 3461], [4131, 3253], [4192, 4162], [1203, 4370]], [[1056, 4203], [4537, 4042], [4579, 4968], [1097, 5129]], [[1801, 5380], [4433, 5253], [4460, 5823], [1828, 5950]]], 'text_det_params': {'limit_side_len': 960, 'limit_type': 'max', 'thresh': 0.3, 'max_side_limit': 4000, 'box_thresh': 0.0, 'unclip_ratio': 1.6}, 'text_type': 'general', 'textline_orientation_angles': [1, 1, 1, 1, 1], 'text_rec_score_thresh': 0.0, 'return_word_box': False, 'rec_texts': ['ERDBEERE', 'PASSIERT-KERNLOS', 'SOFT', 'FEIN&', 'Solbella'], 'rec_scores': [0.9902427792549133, 0.98457479

In [100]:
from PIL import Image
import io
res = response.json()

result_img_b = base64.b64decode(res["result"]["ocrResults"][0]["ocrImage"])
result_img = Image.open(io.BytesIO(result_img_b))
result_img.show()


In [101]:
angle = res["result"]["ocrResults"][0]["prunedResult"]["doc_preprocessor_res"]["angle"]
print("Detected document rotation angle:", angle)

Detected document rotation angle: 180


In [102]:
# Get all textst

result_texts=[]

ocr_results = res["result"]["ocrResults"]

for i in range(len(ocr_results)):
    result_texts.extend(ocr_results[i]["prunedResult"]["rec_texts"])

print(result_texts)

['ERDBEERE', 'PASSIERT-KERNLOS', 'SOFT', 'FEIN&', 'Solbella']
